# 02. 토스광고 CTR 데이터 경량화(Diet Parquet) & Local CV 분할 확정

이 노트북은 약 **9.4 GB** 원천 데이터(`train.parquet`, `test.parquet`)를 **OOM 없이 배치 스트리밍**으로 경량 분리하고, 모든 모델 트랙(GBDT / DeepCTR / Ensemble)이 공유할 **표준 Local CV 분할(`folds.parquet`)**을 확정합니다.

### 핵심 Data Contract 산출물
1. `train_tabular_diet.parquet`: `row_id` (`int64`, 0-based), `clicked` (`int8`), `seq` 제외 정형 다운캐스팅 피처 (`int8~int32`, `float32`)
2. `test_tabular_diet.parquet`: `ID` (원본 순서·타입 보존), `seq` 제외 정형 다운캐스팅 피처 (Train과 동일 dtype)
3. `train_seq.parquet` / `test_seq.parquet`: 가변 길이 문자열 시퀀스(`seq`) 분리 아티팩트 (`row_id`/`ID`, `seq`)
4. `folds.parquet`: 표준 CV 인덱스 (`row_id`, `fold_id`, `is_oot_val`), 결정론적 5-Fold Stratified + OOT 마킹

In [ ]:
from google.colab import drive
import os
import sys
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
import yaml

import os as _os
if not _os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

# TODO: Google Drive 내 프로젝트 루트 경로에 맞게 수정하세요.
PROJECT_ROOT = Path('/content/drive/MyDrive/toss_ctr')
if (PROJECT_ROOT / 'src').exists() and str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))
if (PROJECT_ROOT / 'pyproject.toml').exists():
    os.chdir(PROJECT_ROOT)
    %pip install -e . --no-deps --quiet

from toss_ctr.data import (
    DEFAULT_CONTRACT,
    build_cv_folds,
    build_diet_artifacts,
    collect_parquet_column_stats,
    discover_feature_groups,
    infer_safe_dtypes,
    load_eda_summary,
    load_folds,
    validate_folds_contract,
)

diet_cfg_path = PROJECT_ROOT / 'configs' / 'data_diet.yaml'
cv_cfg_path = PROJECT_ROOT / 'configs' / 'cv_strategy.yaml'

with open(diet_cfg_path, 'r', encoding='utf-8') as f:
    diet_cfg = yaml.safe_load(f)
with open(cv_cfg_path, 'r', encoding='utf-8') as f:
    cv_cfg = yaml.safe_load(f)

print('Loaded data_diet.yaml:', diet_cfg)
print('Loaded cv_strategy.yaml:', cv_cfg)

### [해석 1] 환경 마운트, `toss_ctr` 패키지 로드 및 YAML 설정 확인
- Google Drive 마운트 후 `toss_ctr` 패키지를 editable 모드(`--no-deps`)로 연결해 본선 코드 검증과 동일한 모듈 코드(`src/toss_ctr/data/`)를 직접 실행합니다.
- `configs/data_diet.yaml`의 `batch_size=262144`, `compression=zstd` 설정과 `configs/cv_strategy.yaml`의 `n_splits=5`, `seed=42`, `oot.mode=auto` 설정이 정상 로드되었는지 확인합니다.

In [ ]:
raw_train_path = Path(diet_cfg['paths']['raw_train'])
raw_test_path = Path(diet_cfg['paths']['raw_test'])
eda_summary_path = Path(diet_cfg['paths']['eda_summary'])

train_pf = pq.ParquetFile(raw_train_path)
test_pf = pq.ParquetFile(raw_test_path)

feature_cols = [
    c for c in train_pf.schema_arrow.names
    if c not in (DEFAULT_CONTRACT.target_col, DEFAULT_CONTRACT.seq_col, DEFAULT_CONTRACT.id_col)
]
eda_summary = load_eda_summary(eda_summary_path)
groups = discover_feature_groups(feature_cols, eda_summary=eda_summary)
print('피처 그룹별 컬럼 수:', {k: len(v) for k, v in groups.items()})

# Parquet Row-Group 푸터 메타데이터만 스캔(RAM ~0MB)해 전역 min/max 기반 안전 다운캐스팅 계획을 미리 확인합니다.
col_stats = collect_parquet_column_stats([raw_train_path, raw_test_path], columns=feature_cols)
planned_dtypes = infer_safe_dtypes(
    train_pf.schema_arrow,
    eda_summary=eda_summary,
    column_stats=col_stats,
    downcast_float64_to_float32=diet_cfg['downcast']['downcast_float32'],
)

dtype_summary = pd.DataFrame([
    {
        'column': col,
        'src_dtype': str(train_pf.schema_arrow.field(col).type),
        'target_dtype': str(planned_dtypes.get(col, train_pf.schema_arrow.field(col).type)),
    }
    for col in feature_cols
])
display(dtype_summary.groupby(['src_dtype', 'target_dtype']).size().reset_index(name='col_count'))

### [해석 2] 피처 그룹 동적 분류 및 OOM-Free 다운캐스팅 계획 검증
- `discover_feature_groups()`가 명시 피처(`gender`, `age_group`, `inventory_id`, `day_of_week`, `hour`, `l_feat_14`)와 익명 그룹(`l_feat_*`, `feat_a~e_*`, `history_a_*`)을 상호배타적으로 분류한 결과를 확인합니다.
- `collect_parquet_column_stats()`는 데이터 본문을 로드하지 않고 Train+Test 양쪽 Parquet 푸터 메타데이터의 `min`/`max`만 집계하므로, **Train에는 작고 Test에는 큰 값**이 들어있어 발생하는 정수 오버플로우를 사전에 완벽히 차단합니다.

In [ ]:
diet_manifest = build_diet_artifacts(
    train_path=raw_train_path,
    test_path=raw_test_path,
    output_dir=diet_cfg['paths']['output_dir'],
    batch_size=diet_cfg['streaming']['batch_size'],
    compression=diet_cfg['streaming']['compression'],
    compression_level=diet_cfg['streaming'].get('compression_level'),
    eda_summary_path=eda_summary_path,
    downcast_float32=diet_cfg['downcast']['downcast_float32'],
    downcast_integers=diet_cfg['downcast']['downcast_integers'],
    verify_row_counts=diet_cfg['downcast']['verify_row_counts'],
    save_manifest=diet_cfg['downcast']['save_manifest'],
)

raw_gb = diet_manifest['raw_sizes_bytes']['total'] / (1024 ** 3)
diet_gb = diet_manifest['diet_total_bytes'] / (1024 ** 3)
tab_gb = diet_manifest['tabular_only_bytes'] / (1024 ** 3)

print(f'원천 합계: {raw_gb:.3f} GB')
print(f'전체 Diet(Tabular + Seq): {diet_gb:.3f} GB (절감률: {diet_manifest["total_size_reduction_pct"]:.2f}%)')
print(f'정형 Diet(Tabular Only):  {tab_gb:.3f} GB (원천 대비 절감률: {diet_manifest["tabular_size_reduction_pct"]:.2f}%)')

artifact_df = pd.DataFrame.from_dict(diet_manifest['artifacts'], orient='index')
display(artifact_df)

### [해석 3] 정형 Diet / `seq` 분리 스트리밍 실행 결과 및 용량 절감 분석
- `ParquetWriter`를 통한 Row-Group/배치 스트리밍으로 RAM 피크를 최소화하면서 4개의 표준 산출물(`train_tabular_diet`, `test_tabular_diet`, `train_seq`, `test_seq`)을 생성했습니다.
- `seq` 분리 및 다운캐스팅(`float32`, `int8~int32`)으로 정형 테이블(`tabular_only_bytes`) 용량이 대폭 축소되어, 이후 GBDT 트랙(`tabular-gbdt-engineer`)에서 다중 피처 엔지니어링과 5-Fold 교차검증을 고속·안정적으로 반복할 수 있습니다.

In [ ]:
train_diet_path = Path(diet_manifest['artifacts']['train_tabular']['path'])
test_diet_path = Path(diet_manifest['artifacts']['test_tabular']['path'])
folds_out_path = Path(cv_cfg['paths']['output_path'])
cv_report_path = Path(cv_cfg['paths']['report_path'])

folds_df, cv_summary = build_cv_folds(
    train_path=train_diet_path,
    test_path=test_diet_path,
    output_path=folds_out_path,
    n_splits=cv_cfg['cv']['n_splits'],
    seed=cv_cfg['cv']['seed'],
    oot_mode=cv_cfg['oot']['mode'],
    tvd_threshold=cv_cfg['oot']['tvd_threshold'],
    mark_last_day_if_in_time=cv_cfg['oot']['mark_last_day_if_in_time'],
    batch_size=cv_cfg['streaming']['batch_size'],
    compression=cv_cfg['streaming']['compression'],
    report_path=cv_report_path,
)

print('Temporal Regime:', cv_summary['temporal_regime'])
print(f"TVD: {cv_summary['tvd']:.4f}, Unseen Test Days: {cv_summary['unseen_test_days']}")
print(f"OOT Marked: {cv_summary['oot_marked']} (oot_day={cv_summary['oot_day']}, oot_rows={cv_summary['oot_rows']:,}, ratio={cv_summary['oot_ratio']:.2%})")
print('Stratification Fallback:', cv_summary['stratification_fallback'])

print('\n[Train vs Test day_of_week 분포]')
display(pd.DataFrame(cv_summary['day_of_week_compare']))

print('\n[5-Fold CV 분포 및 클릭률(CTR) 정합성]')
fold_summary_df = pd.DataFrame(cv_summary['fold_summary'])
display(fold_summary_df)

# Fold별 day_of_week 교차표 검증 (경량 컬럼만 로드)
dow_col = pq.read_table(train_diet_path, columns=['day_of_week', 'clicked']).to_pandas()
dow_col['fold_id'] = folds_df['fold_id'].to_numpy()
dow_col['is_oot_val'] = folds_df['is_oot_val'].to_numpy()
display(pd.crosstab(dow_col['fold_id'], dow_col['day_of_week'], normalize='index').round(4))

### [해석 4] 시간축 OOT 감지 및 5-Fold Stratified CV 균형성 검증
- **시간축 진단(`temporal_regime`, `TVD`)**: `train`과 `test` 간 `day_of_week` 분포 편차(Total Variation Distance) 및 미등장 요일 여부를 정량 점검하고, 마지막 요일(`oot_day`)을 `is_oot_val=True`로 마킹해 시간축 분포 변화(Time Shift)에 따른 과적합을 보조 검증할 수 있도록 고정했습니다.
- **Fold 균형성**: `(day_of_week, clicked)` 조합 기준 `StratifiedKFold(seed=42)` 적용 결과, 5개 폴드 간 전체 행 수, 요일 비율, 양성 클릭률(`ctr`)이 소수점 이하 수준까지 균일하게 배분되었음을 확인했습니다.

In [ ]:
# Contract 무결성 및 ID/row_id 순서 보존 최종 스모크 검증
loaded_folds = load_folds(folds_out_path, validate=True, n_rows=diet_manifest['train_rows'], n_splits=cv_cfg['cv']['n_splits'])
contract_check = validate_folds_contract(loaded_folds, n_rows=diet_manifest['train_rows'], n_splits=cv_cfg['cv']['n_splits'])
print('Folds Contract Verification:', contract_check)

# Train/Test 공통 피처 dtype 일치 여부 및 test ID 순서 보존 검사
tab_train_schema = pq.ParquetFile(train_diet_path).schema_arrow
tab_test_schema = pq.ParquetFile(test_diet_path).schema_arrow
shared_cols = [c for c in tab_train_schema.names if c in set(tab_test_schema.names)]
dtype_mismatches = [
    (c, str(tab_train_schema.field(c).type), str(tab_test_schema.field(c).type))
    for c in shared_cols
    if tab_train_schema.field(c).type != tab_test_schema.field(c).type
]
assert not dtype_mismatches, f'Train/Test 공통 컬럼 dtype 불일치: {dtype_mismatches}'
print(f'Train/Test 공통 피처 {len(shared_cols)}개 dtype 100% 일치 확인 완료!')

# Test ID 순서 일치 검증 (원천 vs Diet Tabular vs Seq)
raw_test_id = pq.read_table(raw_test_path, columns=['ID']).column('ID').to_numpy()
diet_test_id = pq.read_table(test_diet_path, columns=['ID']).column('ID').to_numpy()
seq_test_id = pq.read_table(Path(diet_manifest['artifacts']['test_seq']['path']), columns=['ID']).column('ID').to_numpy()
assert (raw_test_id == diet_test_id).all(), 'test_tabular_diet ID 순서가 원천과 다릅니다!'
assert (raw_test_id == seq_test_id).all(), 'test_seq ID 순서가 원천과 다릅니다!'
print(f'Test ID({len(raw_test_id):,}건) 원천/Diet/Seq 순서 100% 일치 확인 완료!')

### [해석 5] Data Contract 무결성 및 ID/dtype 일치 최종 확인
- `validate_folds_contract()`로 `row_id` `0..N-1` 연속성, 결측 0건, `fold_id` `[0, 4]` 범위, `is_oot_val` bool 타입을 모두 검증했습니다.
- Train/Test 공통 피처 117개(정형 피처)의 PyArrow dtype이 100% 동일하며, `test` 1,527,298건의 `ID` 순서가 원천/Diet/Seq 파일 간 완전히 일치함을 확인했습니다.

### [종합 요약 및 다음 단계 (Gate 2) 안내]
- **확정된 데이터 경로**:
  - 정형 학습/추론: `train_tabular_diet.parquet`, `test_tabular_diet.parquet`
  - 시퀀스 학습/추론: `train_seq.parquet`, `test_seq.parquet`
  - 공통 교차검증 분할: `folds.parquet` (`row_id`, `fold_id`, `is_oot_val`)
- **다음 단계**: 이제 정형 GBDT 트랙(`notebooks/03_*`, `04_*`)과 시퀀스 DeepCTR 트랙(`notebooks/05_*`, `06_*`)이 동일한 `row_id`와 `fold_id` 위에서 병렬로 학습·추론을 수행하고 표준 OOF(`row_id, clicked, raw_logit, pred_prob`) / Test(`ID, raw_logit, pred_prob`) 파일을 생성할 수 있습니다.